# Experiment 11: Implementation of the VFDT Algorithm for Data Classification

## Aim

- Data classification process.
- Hoeffding bound.
- Incremental decision tree construction.
- Data pipeline.

## Theory

The Very Fast Decision Tree, also called the Hoeffding Tree, builds a decision tree from a data
stream without storing the stream. Each example is read once, used to update the statistics stored
at a leaf, and then discarded.

The key idea is the **Hoeffding bound**. It states that after n independent observations of a
variable with range R, the true mean differs from the observed mean by at most

```text
epsilon = sqrt( (R^2 * ln(1 / delta)) / (2 * n) )
```

with probability 1 - delta. At a leaf the information gain of every attribute is calculated. If the
gap between the best attribute and the second best attribute is greater than epsilon, then the best
attribute is confidently the winner and the leaf is split. If not, more examples are collected. This
lets the tree grow from a small number of examples with a statistical guarantee, instead of scanning
the whole dataset.

## Implementation Steps

1. Library
2. VFDT Node
3. VFDT Tree
4. Entropy
5. Hoeffding bound
6. Predict
7. Dataset
8. Discretize
9. Accuracy calculation

## Requirements

- Python 3.x
- Jupyter Notebook
- NumPy
- Scikit-learn

In [1]:
import numpy as np
import math
from collections import defaultdict
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
class VFDTNode:

    def __init__(self):
        self.is_leaf = True
        self.children = {}
        self.split_feature = None
        self.class_counts = defaultdict(int)
        self.feature_counts = defaultdict(lambda: defaultdict(int))
        self.samples = 0

**Interpretation**

- Every node starts as a leaf with no children.
- `class_counts` stores how many samples of each class reached the node.
- `feature_counts` stores, for every feature, how often each (value, label) pair was seen.
- `samples` counts the total examples that reached the node.

In [3]:
class VFDT:

    def __init__(self, delta=0.01, min_samples=50):
        self.root = VFDTNode()
        self.delta = delta
        self.min_samples = min_samples
        self.num_features = None

    # Entropy -> compute impurity
    def entropy(self, class_counts):

        total = sum(class_counts.values())

        if total == 0:
            return 0

        entropy = 0

        for c in class_counts.values():
            p = c / total
            if p > 0:
                entropy -= p * math.log2(p)

        return entropy

    def information_gain(self, node, feature):

        total_entropy = self.entropy(node.class_counts)

        feature_values = defaultdict(lambda: defaultdict(int))
        total = node.samples

        for (value, label), count in node.feature_counts[feature].items():
            feature_values[value][label] += count

        weighted_entropy = 0

        for value, counts in feature_values.items():
            sub_total = sum(counts.values())
            weighted_entropy += (sub_total / total) * self.entropy(counts)

        return total_entropy - weighted_entropy

    # Hoeffding bound
    def hoeffding_bound(self, R, delta, n):
        return math.sqrt((R * R * math.log(1 / delta)) / (2 * n))

    def update(self, x, y):

        if self.num_features is None:
            self.num_features = len(x)

        node = self.root

        while not node.is_leaf:
            feature = node.split_feature
            value = int(x[feature])
            if value not in node.children:
                node.children[value] = VFDTNode()
            node = node.children[value]

        node.samples += 1
        node.class_counts[y] += 1

        for f in range(self.num_features):
            value = int(x[f])
            node.feature_counts[f][(value, y)] += 1

        if node.samples >= self.min_samples:
            self.try_split(node)

    def try_split(self, node):

        gains = []

        for f in range(self.num_features):
            gain = self.information_gain(node, f)
            gains.append((gain, f))

        gains.sort(reverse=True)

        if len(gains) < 2:
            return

        best_gain, best_feature = gains[0]
        second_gain, _ = gains[1]

        epsilon = self.hoeffding_bound(1, self.delta, node.samples)

        if best_gain - second_gain > epsilon:
            node.is_leaf = False
            node.split_feature = best_feature
            node.children = {}

    # Predict
    def predict_one(self, x):

        node = self.root

        while not node.is_leaf:
            feature = node.split_feature
            value = int(x[feature])
            if value not in node.children:
                break
            node = node.children[value]

        if len(node.class_counts) == 0:
            return 0

        return max(node.class_counts, key=node.class_counts.get)

    def predict(self, X):

        predictions = []

        for row in X:
            predictions.append(self.predict_one(row))

        return predictions

**Interpretation**

- `entropy()` computes the impurity of a set of class counts.
- `information_gain()` returns the total entropy minus the weighted entropy of the subsets produced
  by a feature.
- `hoeffding_bound()` returns epsilon for the given range R, confidence delta and sample count n.
- `update()` routes one example to a leaf, updates the statistics there and tries to split once the
  leaf has enough samples.
- `try_split()` compares the best and the second best information gain against epsilon and splits
  only when the gap is large enough.
- `predict_one()` walks the tree and returns the majority class of the leaf reached.

In [4]:
X, y = make_classification(
    n_samples=1000,
    n_features=4,
    n_informative=4,
    n_redundant=0,
    random_state=42
)
print("Dataset shape :", X.shape)

Dataset shape : (1000, 4)


In [5]:
X = np.round(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=5
)
print("Training samples :", len(X_train))
print("Testing samples  :", len(X_test))

Training samples : 700
Testing samples  : 300


**Interpretation**

- VFDT splits on discrete attribute values, so the continuous features are rounded to integers.
- 70% of the data is used for training and 30% for testing.

In [6]:
tree = VFDT(delta=0.01, min_samples=40)

for x, label in zip(X_train, y_train):
    tree.update(x, label)

print("Root split feature :", tree.root.split_feature)
print("Number of children :", len(tree.root.children))

Root split feature : 1
Number of children : 10


**Interpretation**

- The examples are fed one at a time, exactly as they would arrive in a stream.
- No example is stored, only the counts at each leaf.

In [7]:
predictions = tree.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy =", accuracy)

Accuracy = 0.76


## Interpretation

- 1000 samples with 4 informative features are generated.
- The features are rounded so that the tree can split on discrete values.
- Each training example is processed once and then discarded.
- A leaf tries to split only after it has seen at least 40 samples.
- The Hoeffding bound decides whether the best attribute is a confident winner.
- The tree splits the root on the winning feature and continues to grow as more data arrives.
- The accuracy obtained on the test data is approximately 0.76.

## Conclusion

- The VFDT (Very Fast Decision Tree) algorithm was successfully implemented using Python.
- It efficiently classified streaming data by constructing the decision tree incrementally.
- The Hoeffding bound helped in selecting the best split without processing the entire dataset.
- The model required less memory and processed large datasets quickly.